In [0]:
import subprocess
import requests 
import tempfile
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")
brev_ip = dbutils.secrets.get(scope="brev", key="brev_ip")
pat = dbutils.secrets.get(scope="databricks", key="pat") 
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None)
email_sender   = dbutils.secrets.get(scope="brev", key="email_sender")
email_password = dbutils.secrets.get(scope="brev", key="email_password")
email_password = email_password.replace('\xa0', '').replace(' ', '').strip()
email_receivers = dbutils.secrets.get(scope="brev", key="email_receivers").split(",")

# Slack webhook
slack_webhook_url = dbutils.secrets.get(scope="brev", key="slack_webhook")

with open("/tmp/ssh_private_key", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key", 0o600)

def check_instance():
    try: 
        result = subprocess.run(
            ["ssh", "-i", "/tmp/ssh_private_key",
             "-o", "StrictHostKeyChecking=no",
             "-o", "ConnectTimeout=10",
             f"ubuntu@{brev_ip}",
             "echo ALIVE"],
            capture_output=True, text=True, timeout=15
        )
        return result.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    except Exception:
        return False

current_job_id = 129636104529500

def check_jobs_running():
    response = requests.get(
        f"{host}/api/2.1/jobs/runs/list",
        headers={"Authorization": f"Bearer {pat}"},
        params={"active_only": True, "limit": 25}
    )
    if response.status_code != 200:
        raise Exception(f"API error: {response.text}")
    
    runs = response.json().get("runs", [])
    active_runs = [
        r for r in runs
        if r.get("state", {}).get("life_cycle_state") == "RUNNING"
        and r.get("job_id") != current_job_id
    ]
    return active_runs

def shutdown_instance():
    result = subprocess.run(
        ["ssh", "-i", "/tmp/ssh_private_key",
         "-o", "StrictHostKeyChecking=no",
         "-o", "ConnectTimeout=10",
         f"ubuntu@{brev_ip}",
         "sudo shutdown -h now"],
        capture_output=True, text=True, timeout=15
    )
    return result.returncode == 0

def send_email(subject: str, body: str):
    msg = MIMEMultipart()
    msg["From"]    = email_sender
    msg["To"]      = ", ".join(email_receivers)
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain", "utf-8"))
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(email_sender, email_password)
        server.sendmail(email_sender, email_receivers, msg.as_string())
    print(f"Email sent to: {email_receivers}")

def send_slack(subject: str, body: str):
    payload = {
        "blocks": [
            {
                "type": "header",
                "text": {"type": "plain_text", "text": subject}
            },
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": f"```{body}```"}
            }
        ]
    }
    resp = requests.post(slack_webhook_url, json=payload)
    if resp.status_code == 200:
        print("Slack message sent")
    else:
        print(f"Slack failed: {resp.status_code} {resp.text}")

def notify(subject: str, body: str):
    send_email(subject, body)
    send_slack(subject, body)

# ── Main logic ───────────────────────────────────────
output_lines = []
instance_up = check_instance()
output_lines.append(f"Instance up: {instance_up}")

if not instance_up:
    output_lines.append("Instance is OFF — nothing to do")
else:
    output_lines.append("Instance is ON — checking Databricks jobs...")
    active_runs = check_jobs_running()
    if active_runs:
        output_lines.append(f"Found {len(active_runs)} active job(s) — instance is busy:")
        for r in active_runs:
            output_lines.append(f"  - {r.get('run_name', 'unknown')} | job_id={r['job_id']} | state={r['state']['life_cycle_state']}")
    else:
        output_lines.append("Instance is ON but no jobs running — shutting down...")
        shutdown_instance()
        output_lines.append("Instance shut down.")

for line in output_lines:
    print(line)

notify(
    subject="[Brev Monitor] Instance Status Report",
    body="\n".join(output_lines)
)